# Introdução ao OpenAI Agents SDK

Este notebook apresenta, de forma progressiva, os principais conceitos do **OpenAI Agents SDK para Python**: agentes, `Runner`, function tools, hosted tools, handoffs, agentes como ferramentas, guardrails, sessões e um projeto integrador.

Os exemplos foram organizados para uso no Google Colab. Execute as células na ordem e mantenha a chave `OPENAI_API_KEY` nos *Secrets* do Colab — nunca diretamente no código.


## 0. Preparação do ambiente

O pacote `openai-agents` contém o SDK. O pacote `openai` é instalado como dependência e será usado diretamente apenas na preparação de arquivos e vector stores. Reinicie o ambiente se o Colab solicitar após a instalação.


In [1]:
%pip install -q -U openai-agents


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.5/968.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00


### Configuração segura da chave de API

No Colab, abra **Secrets**, crie o segredo `OPENAI_API_KEY` e permita que o notebook o acesse. A validação abaixo informa claramente quando a configuração está ausente, sem exibir a chave.


In [2]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Adicione OPENAI_API_KEY aos Secrets do Google Colab.")

os.environ["OPENAI_API_KEY"] = api_key
print("Chave configurada com segurança.")


Chave configurada com segurança.


## 1. Primeiro agente: `Agent` e `Runner`

Um `Agent` combina nome, instruções, modelo e capacidades. O `Runner` executa o ciclo do agente até obter uma resposta final. Em notebooks, usamos `await Runner.run(...)`; em scripts Python comuns, pode-se criar uma função `async main()` e chamá-la com `asyncio.run(main())`.


In [3]:
from agents import Agent, Runner

assistente = Agent(
    name="Assistente didático",
    instructions=(
        "Responda em português brasileiro, com linguagem clara e um exemplo curto."
    ),
    model="gpt-4o-mini",
)

resultado = await Runner.run(
    assistente,
    "Explique recursão em programação em até cinco linhas.",
)

print(resultado.final_output)


Recursão é uma técnica em programação onde uma função se chama a si mesma para resolver um problema. Ela é usada para dividir problemas complexos em subproblemas menores e mais simples. Um exemplo clássico é o cálculo do fatorial de um número, em que o fatorial de \( n \) é \( n \times (n-1)! \). Assim, a função se chama repetidamente até atingir um caso base, geralmente onde a solução é conhecida. É importante garantir que a recursão tenha um caso base que evite chamadas infinitas.


In [4]:
resultado


RunResult(input='Explique recursão em programação em até cinco linhas.', new_items=[MessageOutputItem(agent=Agent(name='Assistente didático', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Responda em português brasileiro, com linguagem clara e um exemplo curto.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None, prompt_cache_options=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=ResponseOutputMessage(id='msg_0c3667e32a0

## 2. Function tools

Uma **function tool** transforma uma função Python em uma ferramenta que o modelo pode escolher e chamar. Type hints e docstrings são importantes porque ajudam o SDK a gerar o esquema da ferramenta e explicam ao modelo quando e como utilizá-la.


### 2.1 Exemplo: consulta de clima com validação

O agente não deve inventar dados meteorológicos. A ferramenta consulta uma API externa, usa `timeout`, valida a resposta e devolve dados estruturados. O modelo fica responsável por converter a pergunta em uma chamada e explicar o resultado ao usuário.


In [27]:
import requests
from agents import Agent, Runner, function_tool


@function_tool
def consultar_clima(cidade: str) -> dict:
    """Obtém o clima atual de uma cidade usando a API Open-Meteo."""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": cidade, "count": 1, "language": "pt", "format": "json"},
        timeout=15,
    )
    geo.raise_for_status()
    locais = geo.json().get("results", [])
    if not locais:
        return {"erro": f"Cidade não encontrada: {cidade}"}

    local = locais[0]
    clima = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": local["latitude"],
            "longitude": local["longitude"],
            "current": "temperature_2m,apparent_temperature,wind_speed_10m",
            "timezone": "auto",
        },
        timeout=15,
    )
    clima.raise_for_status()
    atual = clima.json()["current"]

    return {
        "cidade": local["name"],
        "estado": local.get("admin1"),
        "temperatura_c": atual["temperature_2m"],
        "sensacao_c": atual["apparent_temperature"],
        "vento_kmh": atual["wind_speed_10m"],
        "horario_local": atual["time"],
    }


agente_clima = Agent(
    name="Agente de clima",
    instructions=(
        "Use consultar_clima para perguntas meteorológicas. "
        "Informe a cidade, o horário dos dados e as unidades. Não invente valores."
    ),
    tools=[consultar_clima],
    model="gpt-4o-mini",
)

resultado = await Runner.run(agente_clima, "Como está o clima na paulista em são paulo?")
print(resultado.final_output)


Atualmente, o clima na Avenida Paulista, em São Paulo, é o seguinte:

- **Temperatura:** 19,8 °C
- **Sensação Térmica:** 20,1 °C
- **Vento:** 10,2 km/h

Esses dados são de agora, no horário local.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
resultado

RunResult(input='Como está o clima agora em Recife?', new_items=[ToolCallItem(agent=Agent(name='Agente de clima', handoff_description=None, tools=[FunctionTool(name='consultar_clima', description='Obtém o clima atual de uma cidade usando a API Open-Meteo.', params_json_schema={'properties': {'cidade': {'title': 'Cidade', 'type': 'string'}}, 'required': ['cidade'], 'title': 'consultar_clima_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7ec0f76a2810>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)], mcp_servers=[], mcp_config={}, instructions='Use consultar_clima para perguntas meteorológicas. Informe a cidade, o horário dos dados e as unidades. Não 

## 3. Hosted tools

**Hosted tools** são executadas na infraestrutura da OpenAI. Diferentemente de uma function tool, seu programa não implementa a execução da ferramenta. Elas exigem um modelo OpenAI compatível com a **Responses API**, caminho usado por padrão pelo SDK para modelos OpenAI.

Nesta seção veremos `WebSearchTool`, `FileSearchTool` e `CodeInterpreterTool`.


### 3.1 `WebSearchTool`: informações atuais

`WebSearchTool` permite pesquisar informações recentes na web. A instrução exige fontes e datas, reduzindo o risco de apresentar fatos desatualizados como atuais.


In [7]:
from agents import Agent, Runner, WebSearchTool

agente_pesquisa = Agent(
    name="Pesquisador web",
    instructions=(
        "Pesquise a web quando a pergunta depender de informação atual. "
        "Diferencie a data de publicação da data do acontecimento, cite as fontes "
        "e diga quando houver informações conflitantes."
    ),
    tools=[WebSearchTool(search_context_size="medium")],
    model="gpt-4o-mini",
)

resultado = await Runner.run(
    agente_pesquisa,
    "Pesquise uma notícia recente sobre transporte público em São Paulo e resuma em três tópicos.",
)
print(resultado.final_output)


Recentes notícias sobre o transporte público em São Paulo destacam os seguintes pontos:

1. **Reajuste das tarifas de transporte público**: Em dezembro de 2025, a Prefeitura de São Paulo anunciou um aumento nas tarifas de ônibus municipais, que passaram de R$ 5,00 para R$ 5,30, e nos trens e metrôs, que subiram de R$ 5,20 para R$ 5,40. O reajuste entrou em vigor em janeiro de 2026, visando cobrir o aumento nos custos operacionais, como energia, manutenção da frota e folha de pagamento. ([noticias.uol.com.br](https://noticias.uol.com.br/ultimas-noticias/agencia-estado/2025/12/29/prefeitura-e-governo-de-sp-anunciam-reajuste-da-tarifa-de-onibus-metro-e-trens.amp.htm?utm_source=openai))

2. **Operação Última Parada e investigação de empresas de ônibus**: Em junho de 2026, a Operação Última Parada foi deflagrada para investigar a infiltração do Primeiro Comando da Capital (PCC) no sistema de transporte público de São Paulo. A operação resultou em mandados de prisão e busca em diferentes cid

### 3.2 `FileSearchTool`: visão geral

`FileSearchTool` realiza busca semântica em documentos previamente indexados em um **vector store**. O fluxo tem duas partes:

1. preparar e enviar os arquivos ao vector store;
2. fornecer o ID desse vector store ao agente.

O exemplo cria um arquivo didático pequeno. Em um projeto real, substitua-o por PDFs, documentos ou textos da sua base de conhecimento.


#### Etapa 1 — criar o documento e o vector store

O cliente `OpenAI` administra o upload. `upload_and_poll` aguarda a indexação terminar, evitando que o agente pesquise antes de o conteúdo estar pronto. Execute esta preparação uma vez e reutilize o `vector_store.id` nas execuções seguintes.


In [8]:
from pathlib import Path
from openai import OpenAI

politica = Path("/content/politica.txt")
politica.write_text(
    "Política de trocas da TechMais\n"
    "- Produtos sem defeito podem ser trocados em até 7 dias corridos.\n"
    "- É necessário apresentar o comprovante de compra.\n"
    "- Produtos com defeito passam por avaliação técnica.\n"
    "- O prazo de avaliação técnica é de até 5 dias úteis.\n"
    "- Itens com dano causado por mau uso não são cobertos.\n",
    encoding="utf-8",
)

client = OpenAI()
vector_store = client.vector_stores.create(name="Base de conhecimento Prova")

with politica.open("rb") as arquivo:
    arquivo_indexado = client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id,
        file=arquivo,
    )

print("Vector store:", vector_store.id)
print("Status da indexação:", arquivo_indexado.status)


Vector store: vs_6a76648e14988191a4367b9115cccf88
Status da indexação: completed


#### Etapa 2 — responder com base no documento

`max_num_results` limita a quantidade de trechos recuperados. `include_search_results=True` mantém os resultados da busca na resposta bruta, o que ajuda em auditoria e depuração. As instruções orientam o agente a admitir quando a base não contém a resposta.


In [9]:
from agents import Agent, FileSearchTool, Runner

agente_documentos = Agent(
    name="Assistente aluno de python",
    instructions=(
        "Responda somente com base nos documentos encontrados. "
        "Informe o documento usado. Se a base não contiver a resposta, diga isso claramente."
    ),
    tools=[
        FileSearchTool(
            vector_store_ids=[vector_store.id],
            max_num_results=5,
            include_search_results=True,
        )
    ],
    model="gpt-4o-mini",
)

resultado = await Runner.run(
    agente_documentos,
    """Com quantos dias posso trocar?""",
)
print(resultado.final_output)


Você pode trocar produtos sem defeito em até 7 dias corridos, desde que apresente o comprovante de compra.


In [10]:
resultado

RunResult(input='Com quantos dias posso trocar?', new_items=[ToolCallItem(agent=Agent(name='Assistente aluno de python', handoff_description=None, tools=[FileSearchTool(vector_store_ids=['vs_6a76648e14988191a4367b9115cccf88'], max_num_results=5, include_search_results=True, ranking_options=None, filters=None)], mcp_servers=[], mcp_config={}, instructions='Responda somente com base nos documentos encontrados. Informe o documento usado. Se a base não contiver a resposta, diga isso claramente.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None, prompt_cache

#### Exemplo adicional — comparar duas regras do documento

Uma boa pergunta de recuperação exige combinar trechos relacionados. Aqui, o agente diferencia troca sem defeito de avaliação de produto defeituoso e evita misturar os dois prazos.


In [11]:
resultado = await Runner.run(
    agente_documentos,
    "Compare o processo de troca sem defeito com o processo de produto defeituoso. Organize em tabela.",
)
print(resultado.final_output)


Aqui está a comparação entre o processo de troca sem defeito e o processo de produto defeituoso, organizada em tabela:

| Aspecto                     | Troca Sem Defeito                     | Troca de Produto Defeituoso            |
|-----------------------------|---------------------------------------|-----------------------------------------|
| Prazo para troca            | Até 7 dias corridos                  | Até 5 dias úteis para avaliação        |
| Necessidade de comprovante  | Sim                                   | Sim                                     |
| Natureza do produto         | Produto em boas condições             | Produto passa por avaliação técnica     |
| Itens não cobertos          | N/A                                   | Dano causado por mau uso não é coberto |

As informações foram extraídas do documento sobre a política de trocas da TechMais.


#### Exemplo adicional — testar ausência de informação

Testar perguntas fora da base é uma prática importante. O comportamento esperado é declarar que a informação não foi encontrada, em vez de preencher a lacuna com conhecimento geral ou suposição.


In [12]:
resultado = await Runner.run(
    agente_documentos,
    "Qual é o telefone da loja e qual é o horário de atendimento?",
)
print(resultado.final_output)


Não encontrei informações sobre o telefone da loja ou o horário de atendimento nos documentos disponíveis.


# Exercício

Recriar o RAG usando o agents sdk

In [29]:
from pathlib import Path
from openai import OpenAI
from agents import Agent, FileSearchTool, Runner

# criando o agente
manual = Path("/content/ManualCandidatoFIAP.pdf")

client = OpenAI()
vector_store = client.vector_stores.create(name="Manual Candidato da FIAP")

with manual.open("rb") as arquivo:
    arquivo_indexado = client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id,
        file=arquivo,
    )

# fazendo a parte da resposta
agente_documentos = Agent(
    name="Assistente do Candidato",
    instructions=(
        "Responda somente com base nos documentos encontrados. "
        "Informe o documento usado. Se a base não contiver a resposta, diga isso claramente."
    ),
    tools=[
        FileSearchTool(
            vector_store_ids=[vector_store.id],
            max_num_results=5,
            include_search_results=True,
        )
    ],
    model="gpt-4o-mini",
)

resultado = await Runner.run(
    agente_documentos,
    "Quando sai a lista de aprovados?",
)
print(resultado.final_output)



A lista de aprovados será divulgada a partir de 08/12/2025, às 18h, na internet, no site www.fiap.com.br.


### 3.3 `CodeInterpreterTool`: visão geral

`CodeInterpreterTool` permite ao modelo escrever e executar Python em um contêiner isolado. É útil para cálculos, estatística, transformação de dados e geração de gráficos. Com `container={"type": "auto"}`, a infraestrutura cria o ambiente automaticamente para a execução.

O resultado deve ser verificado: executar código reduz erros aritméticos, mas não garante que o método estatístico escolhido seja adequado.


#### Exemplo 1 — cálculo verificável

As instruções obrigam o agente a usar Python e apresentar fórmula, substituição e resultado. Isso torna o raciocínio numérico mais auditável para os alunos.


In [13]:
from agents import Agent, CodeInterpreterTool, Runner

agente_calculos = Agent(
    name="Analista quantitativo",
    instructions=(
        "Use o Code Interpreter em todos os cálculos. "
        "Mostre a fórmula, os valores substituídos e o resultado arredondado."
    ),
    tools=[
        CodeInterpreterTool(
            tool_config={
                "type": "code_interpreter",
                "container": {"type": "auto"},
            }
        )
    ],
    model="gpt-4o-mini",
)

resultado = await Runner.run(
    agente_calculos,
    "Uma loja vendeu 128 unidades contra um concorrente que vendeu 80. Calcule o lift percentual.",
)
print(resultado.final_output)


O cálculo do lift percentual pode ser feito usando a seguinte fórmula:

\[
\text{Lift Percentual} = \left( \frac{\text{Vendas da Loja} - \text{Vendas do Concorrente}}{\text{Vendas do Concorrente}} \right) \times 100
\]

Substituindo os valores:

- Vendas da Loja = 128
- Vendas do Concorrente = 80

Vamos calcular isso agora.


#### Exemplo 2 — análise estatística de vendas

O agente recebe uma pequena série de vendas, calcula estatísticas descritivas e identifica observações acima de média mais um desvio padrão. Pedir método e valores intermediários facilita conferir a resposta.


In [14]:
vendas = [82, 79, 91, 88, 84, 120, 86, 83, 95, 89, 130, 87]

prompt_analise = (
    f"Analise as vendas semanais: {vendas}\n"
    "1. Calcule média, mediana, desvio padrão amostral e coeficiente de variação.\n"
    "2. Identifique valores acima de média + 1 desvio padrão.\n"
    "3. Explique em linguagem simples o que os resultados sugerem.\n"
    "Use Python e mostre os valores utilizados."
)

resultado = await Runner.run(agente_calculos, prompt_analise)
print(resultado.final_output)


Aqui estão os resultados das análises:

1. **Cálculos**:
   - **Média**: 
     \[
     \text{Média} = \frac{\Sigma \text{vendas}}{n} = \frac{82 + 79 + 91 + 88 + 84 + 120 + 86 + 83 + 95 + 89 + 130 + 87}{12} \approx 92.83
     \]
   - **Mediana**: 
     Os dados ordenados são: [79, 82, 83, 84, 86, 87, 88, 89, 91, 95, 120, 130], portanto a mediana é 87.5.
   - **Desvio padrão amostral**: 
     \[
     s = \sqrt{\frac{\Sigma (x_i - \text{média})^2}{n-1}} \approx 15.75
     \]
   - **Coeficiente de variação**: 
     \[
     \text{Coef. Var.} = \left(\frac{s}{\text{média}}\right) \times 100 \approx 16.97\%
     \]

2. **Limite superior**: 
   \[
   \text{Limite Superior} = \text{Média} + 1 \times \text{Desvio Padrão} \approx 108.59
   \]
   - **Valores acima do limite**: [120, 130]

3. **Interpretação**:
   - A média de vendas é de aproximadamente 92.83, indicando o desempenho médio dos dados.
   - A mediana de 87.5 sugere que metade das semanas teve vendas abaixo desse valor e metade acima.

In [15]:
resultado.raw_responses[0]

ModelResponse(output=[ResponseCodeInterpreterToolCall(id='ci_074f6231b3dae6dd006a7664a698a88191a09e6dcaf516e798', code='import numpy as np\r\nimport statistics as stats\r\n\r\n# Dados de vendas semanais\r\nvendas = [82, 79, 91, 88, 84, 120, 86, 83, 95, 89, 130, 87]\r\n\r\n# 1. Cálculo da média, mediana, desvio padrão amostral e coeficiente de variação\r\nmedia = np.mean(vendas)\r\nmediana = np.median(vendas)\r\ndesvio_padrao = np.std(vendas, ddof=1)  # amostral\r\ncoef_variacao = (desvio_padrao / media) * 100  # em porcentagem\r\n\r\n# 2. Identificação de valores acima da média + 1 desvio padrão\r\nlimite_superior = media + desvio_padrao\r\nvalores_acima_limite = [venda for venda in vendas if venda > limite_superior]\r\n\r\n(media, mediana, desvio_padrao, coef_variacao, limite_superior, valores_acima_limite)', container_id='cntr_6a7664a628bc819192f23ad9cb2c4f3c020492d45d86d5db', outputs=None, status='completed', type='code_interpreter_call'), ResponseOutputMessage(id='msg_074f6231b3dae

#### Exemplo 3 — analisar um CSV no contêiner

Para arquivos reais, crie um contêiner, envie o CSV a ele e passe o ID do contêiner à ferramenta. Assim, o agente pode carregar o arquivo com pandas. Esta célula cria uma base fictícia para manter o exemplo reproduzível.


In [16]:
import pandas as pd
from openai import OpenAI

pd.DataFrame(
    {
        "produto": ["A", "B", "C", "D"],
        "venda_media": [120, 75, 210, 95],
        "desvio_vendas": [12, 30, 18, 9],
        "margem": [0.18, 0.25, 0.12, 0.22],
    }
).to_csv("vendas_produtos.csv", index=False)

client = OpenAI()
container = client.containers.create(name="analise-vendas-aula")

with open("vendas_produtos.csv", "rb") as arquivo:
    client.containers.files.create(container_id=container.id, file=arquivo)

print("Contêiner preparado:", container.id)


Contêiner preparado: cntr_6a7664af98dc8191b7a397c0ed43de16099da85e546a68d3


O agente abaixo reutiliza o contêiner que contém o CSV. A instrução especifica as colunas e exige uma recomendação baseada em critérios explícitos: média alta e variabilidade relativa baixa.


In [17]:
agente_csv = Agent(
    name="Analista de vendas",
    instructions=(
        "Use Python e pandas para analisar vendas_produtos.csv. "
        "Mostre os cálculos, confira valores ausentes e não invente colunas."
    ),
    tools=[
        CodeInterpreterTool(
            tool_config={
                "type": "code_interpreter",
                "container": container.id,
            }
        )
    ],
    model="gpt-4o-mini",
)

resultado = await Runner.run(
    agente_csv,
    "Calcule o coeficiente de variação das vendas e recomende o produto mais estável entre os de venda média acima de 90.",
)
print(resultado.final_output)


Aqui estão os resultados da análise das vendas:

### Coeficiente de Variação dos Produtos com Venda Média Acima de 90:
| Produto | Coeficiente de Variação (%) |
|---------|------------------------------|
| A       | 10.00                        |
| C       | 8.57                         |
| D       | 9.47                         |

### Produto Mais Estável
O produto mais estável, com um coeficiente de variação mais baixo, é o **Produto C**, que possui:
- **Venda Média:** 210
- **Desvio de Vendas:** 18
- **Coeficiente de Variação:** 8.57%

Se você precisar de mais análises ou informações, é só avisar!


### 3.4 Combinando `FileSearchTool` e `CodeInterpreterTool`

Um único agente pode recuperar regras de documentos e executar cálculos. O ponto crítico é definir responsabilidades: a busca fornece os fatos; o interpretador executa a matemática; o agente integra os resultados e explicita qualquer ausência de informação.


In [18]:
agente_hibrido = Agent(
    name="Analista de políticas e dados",
    instructions=(
        "Use File Search para recuperar regras da empresa e Code Interpreter para cálculos. "
        "Não trate uma regra calculada como se estivesse escrita no documento. "
        "Informe claramente a origem de cada conclusão."
    ),
    tools=[
        FileSearchTool(vector_store_ids=[vector_store.id], max_num_results=5),
        CodeInterpreterTool(
            tool_config={
                "type": "code_interpreter",
                "container": {"type": "auto"},
            }
        ),
    ],
    model="gpt-4o-mini",
)

resultado = await Runner.run(
    agente_hibrido,
    "Recupere o prazo de avaliação técnica. Se 37 solicitações forem distribuídas igualmente por esse número de dias úteis, calcule a média diária.",
)
print(resultado.final_output)


O prazo de avaliação técnica é de **até 5 dias úteis**. Distribuindo 37 solicitações igualmente por esse período, a média diária de solicitações é de **7,4 solicitações por dia**.


## 4. Integrando várias ferramentas

Ferramentas locais e hospedadas podem coexistir. Descrições específicas e instruções claras ajudam o modelo a escolher corretamente. Neste exemplo, clima vem da API; notícias atuais vêm da web; a sugestão de atividade é produzida por uma regra Python determinística.


In [19]:
@function_tool
def sugerir_atividade(cidade: str, temperatura_c: float) -> str:
    """Sugere uma atividade com base na temperatura atual."""
    if temperatura_c >= 30:
        return f"Em {cidade}, priorize atividade aquática e hidratação."
    if temperatura_c >= 20:
        return f"Em {cidade}, uma caminhada ao ar livre pode ser adequada."
    return f"Em {cidade}, considere museu, cinema ou café."


agente_multitool = Agent(
    name="Guia urbano",
    instructions=(
        "Consulte o clima antes de sugerir atividade. "
        "Use busca web somente se o usuário pedir eventos ou notícias atuais. "
        "Informe quais dados sustentam a recomendação."
    ),
    tools=[
        consultar_clima,
        sugerir_atividade,
        WebSearchTool(search_context_size="low"),
    ],
    model="gpt-4o-mini",
)

resultado = await Runner.run(
    agente_multitool,
    "Qual é o clima em Recife e que tipo de atividade você recomenda?",
)
print(resultado.final_output)


Atualmente, em Recife, a temperatura é de **25,2°C**, com uma sensação térmica de **29,2°C**. O vento está a **4,1 km/h**.

Dado o clima agradável, recomendo uma **caminhada ao ar livre**. Isso permitirá aproveitar o tempo agradável e explorar a beleza da cidade!


## 5. Delegação entre agentes

Há dois padrões principais:

- **Handoff:** o especialista assume a execução e se torna o agente ativo.
- **Agent as a tool:** o agente principal mantém o controle, consulta especialistas como ferramentas e compõe a resposta final.

A escolha depende de quem deve ser responsável pela resposta final.


### 5.1 Handoff

O agente de triagem identifica a intenção e transfere o atendimento ao especialista. `handoff_description` esclarece ao agente de triagem quando cada destino deve ser escolhido. `result.last_agent` permite verificar quem concluiu a execução.


In [20]:
from agents import Agent, Runner

agente_cobranca = Agent(
    name="Especialista em cobrança",
    handoff_description="Use para segunda via, vencimento, cobrança duplicada e pagamento.",
    instructions="Resolva dúvidas de cobrança com objetividade. Peça dados mínimos necessários.",
    model="gpt-4o-mini",
)

agente_reembolso = Agent(
    name="Especialista em reembolso",
    handoff_description="Use para cancelamentos, devoluções e solicitação de reembolso.",
    instructions="Explique requisitos, prazos e próximos passos para reembolso.",
    model="gpt-4o-mini",
)

agente_triagem = Agent(
    name="Triagem",
    instructions="Entenda a solicitação e faça handoff ao especialista adequado.",
    handoffs=[agente_cobranca, agente_reembolso],
    model="gpt-4o-mini",
)

resultado = await Runner.run(agente_triagem, "Fui cobrado duas vezes pela mesma compra.")
print(resultado.final_output)
print("Agente que concluiu:", resultado.last_agent.name)


Para ajudar com a cobrança duplicada, por favor, informe:

1. O nome do estabelecimento.
2. A data da compra.
3. O valor cobrado.

Com esses dados, poderei direcionar sua solicitação.
Agente que concluiu: Especialista em cobrança


### 5.2 Agents as tools

Aqui o coordenador chama dois especialistas, mas continua responsável pela resposta. Esse padrão é útil quando o resultado final precisa integrar perspectivas diferentes em um único texto.


In [21]:
especialista_tecnico = Agent(
    name="Especialista técnico",
    instructions="Analise riscos técnicos e proponha uma solução curta.",
    model="gpt-4o-mini",
)

especialista_negocio = Agent(
    name="Especialista de negócio",
    instructions="Analise impacto, custo, benefício e prioridade.",
    model="gpt-4o-mini",
)

coordenador = Agent(
    name="Coordenador",
    instructions=(
        "Consulte os dois especialistas. Compare as recomendações e produza "
        "uma decisão final com justificativa e próximos passos."
    ),
    tools=[
        especialista_tecnico.as_tool(
            tool_name="consultar_tecnico",
            tool_description="Avalia arquitetura, implementação e riscos técnicos.",
        ),
        especialista_negocio.as_tool(
            tool_name="consultar_negocio",
            tool_description="Avalia impacto comercial, custo e prioridade.",
        ),
    ],
    model="gpt-4o-mini",
)

resultado = await Runner.run(
    coordenador,
    "Devemos adicionar busca semântica à central de atendimento?",
)
print(resultado.final_output)


### Análise e Recomendações sobre Implementar Busca Semântica na Central de Atendimento

**1. Considerações Técnicas:**
- **Compatibilidade e Integração:** A busca semântica pode não ser facilmente compatível com os sistemas existentes, o que pode levar a desafios de integração.
- **Custo Inicial Elevado:** O investimento pode ser significativo, sendo necessário justificar os custos com eficiência aprimorada.
- **Treinamento e Adaptação:** Implementar essa tecnologia pode exigir treinamento da equipe, impactando o tempo e os custos.

**2. Aspectos Comerciais:**
- **Impacto na Satisfação do Cliente:** A melhoria na precisão das informações aumenta a satisfação e a fidelização do cliente.
- **Eficiência no Atendimento:** A redução do tempo médio de atendimento pode ser um benefício direto, com melhorias nas métricas de desempenho.
- **Investimento e Retorno:** Apesar dos custos, o potencial aumento na retenção e receita pode justificar a implementação.

### Decisão Final

**Recomendação:

## 6. Guardrails

Guardrails validam entrada ou saída e podem interromper a execução por meio de um **tripwire**. Eles devem complementar — e não substituir — autenticação, autorização, validação determinística e supervisão humana em ações sensíveis.

Com `run_in_parallel=False`, o input guardrail termina antes da chamada principal ao modelo, evitando iniciar o agente quando a entrada já deve ser bloqueada.


In [22]:
from typing import Union
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    Runner,
    TResponseInputItem,
    input_guardrail,
    output_guardrail,
)


@input_guardrail(run_in_parallel=False)
def bloquear_dados_sensiveis(
    ctx,
    agent,
    user_input: Union[str, list[TResponseInputItem]],
) -> GuardrailFunctionOutput:
    texto = user_input if isinstance(user_input, str) else str(user_input)
    termos = ("senha", "cpf", "cartão de crédito")
    encontrou = any(termo in texto.lower() for termo in termos)
    return GuardrailFunctionOutput(
        output_info="Dado sensível detectado" if encontrou else "Entrada permitida",
        tripwire_triggered=encontrou,
    )


@output_guardrail
def bloquear_segredo_na_saida(ctx, agent, agent_output: str) -> GuardrailFunctionOutput:
    encontrou = "SEGREDO_INTERNO" in agent_output
    return GuardrailFunctionOutput(
        output_info="Segredo interno detectado" if encontrou else "Saída permitida",
        tripwire_triggered=encontrou,
    )


agente_suporte = Agent(
    name="Suporte",
    instructions="Responda dúvidas gerais de suporte sem solicitar dados sensíveis.",
    input_guardrails=[bloquear_dados_sensiveis],
    output_guardrails=[bloquear_segredo_na_saida],
    model="gpt-4o-mini",
)


### Testando o guardrail de entrada

Capturar a exceção específica permite tratar o bloqueio de forma diferente de uma falha de rede ou de programação. Em produção, registre apenas metadados necessários e não grave o dado sensível integralmente.


In [23]:
try:
    resultado = await Runner.run(
        agente_suporte,
        "Meu CPF é 123.456.789-00. Consulte meu pedido.",
    )
    print(resultado.final_output)
except InputGuardrailTripwireTriggered:
    print("Solicitação bloqueada: remova os dados sensíveis e tente novamente.")


Solicitação bloqueada: remova os dados sensíveis e tente novamente.


## 7. Memória com sessões

Uma sessão armazena automaticamente o histórico entre execuções. O identificador deve representar de forma estável o usuário ou a conversa. Em aplicações reais, use identificadores não sensíveis, controle acesso ao armazenamento e defina políticas de retenção.


In [24]:
from agents import Agent, Runner, SQLiteSession

agente_memoria = Agent(
    name="Tutor com memória",
    instructions="Use o contexto anterior e seja conciso.",
    model="gpt-4o-mini",
)

sessao = SQLiteSession("aula_usuario_123")

resposta_1 = await Runner.run(
    agente_memoria,
    "Estou planejando visitar o Taj Mahal.",
    session=sessao,
)
print("Resposta 1:", resposta_1.final_output)

resposta_2 = await Runner.run(
    agente_memoria,
    "Em que país ele fica?",
    session=sessao,
)
print("Resposta 2:", resposta_2.final_output)


Resposta 1: Excelente escolha! O Taj Mahal é uma obra-prima da arquitetura. Considere visitar ao amanhecer ou ao entardecer para fotos incríveis e menos multidões. Lembre-se de comprar ingressos com antecedência e verifique as regras de visitação. Boa viagem!
Resposta 2: O Taj Mahal fica na Índia, na cidade de Agra.


## 9. Boas práticas e próximos passos

- Escreva instruções e descrições de ferramentas específicas.
- Use type hints, docstrings, validação de argumentos e tratamento de erros.
- Não coloque chaves, senhas ou dados pessoais no código ou nos prompts.
- Restrinja ferramentas ao menor privilégio necessário.
- Para File Search, avalie a qualidade da recuperação e teste perguntas sem resposta.
- Para Code Interpreter, confira premissas, método, unidades e resultados.
- Use guardrails e supervisão humana em ações sensíveis ou irreversíveis.
- Inspecione traces e crie avaliações automatizadas antes de colocar o agente em produção.

Documentação oficial: [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/) e [Tools](https://openai.github.io/openai-agents-python/tools/).
